In [8]:
import os, shutil, sys

# /kaggle/input me lab8_llm.py dhoondo aur working dir me copy karo
found = False
for root, _, files in os.walk("/kaggle/input"):
    if "lab8_llm.py" in files:
        shutil.copy(os.path.join(root, "lab8_llm.py"), "/kaggle/working/lab8_llm.py")
        print("Copied from:", root)
        found = True
        break

if not found:
    print("lab8_llm.py nahi mila — pehle Kaggle Dataset/Upload se add karo")

# working dir me chale jao taake import kaam kare
os.chdir("/kaggle/working")
sys.path.insert(0, "/kaggle/working")
print("CWD:", os.getcwd())
print("Files:", [f for f in os.listdir(".") if f.endswith(".py")])

Copied from: /kaggle/input/datasets/ahmedhassanofficial/lab8-files
CWD: /kaggle/working
Files: ['lab8_llm.py']


In [9]:
import os
print("LAB_LLM =", os.environ.get("LAB_LLM", "(not set → default = mock)"))

LAB_LLM = (not set → default = mock)


In [12]:
import os
os.environ.pop("LAB_LLM", None)   # mock pe wapas
os.environ.pop("LAB_MODEL", None)

In [18]:
import os
print("LAB_LLM:", os.environ.get("LAB_LLM", "(default=mock)"))
print("Files:", os.listdir("/kaggle/working"))

import lab8_llm as llm
from lab8_llm import ask
llm.set_seed(0)
r = ask("Say hello in one short sentence.")
print("REPLY:", r)
print("calls:", len(llm.LOG))

LAB_LLM: (default=mock)
Files: ['lab8_llm.py', '.virtual_documents', '__pycache__']
REPLY: Hello from the Lab 8 stand-in. I am a Python function, not a language model.
calls: 2


**Activity #02**

In [19]:
QUESTION = "Why does the game crash when the player opens the inventory?"

CONTEXT = {
    "log":     "Error log: " + llm.LOG_TEXT.replace("\n", " "),
    "script":  "Script line: 42: label.text = slot.item.name;",
    "version": "Engine version: Unity 2022.3 LTS",
}

def build_prompt(question, items):
    parts = [CONTEXT[name] for name in items]
    return "\n".join(parts + ["Question: " + question])

# NOTE: manual me typo tha (missing ']'). Sahi hai:
for items in [[], ["log"]]:
    prompt = build_prompt(QUESTION, items)
    reply  = ask(prompt)
    print(f"context {items or 'none'}: {len(prompt)} characters sent")
    print("  reply:", reply)

context none: 70 characters sent
  reply: I cannot be sure without more information. Common causes of a crash when a menu opens are null references, missing assets or endless loops.
context ['log']: 223 characters sent
  reply: Likely cause: a NullReferenceException at InventoryUI.cs:42, so something on that line is empty.


In [20]:
p = build_prompt(QUESTION, ["log"])
print("Prompt length:", len(p))
print("----- BEGIN PROMPT -----")
print(p)
print("----- END PROMPT -----")

Prompt length: 223
----- BEGIN PROMPT -----
Error log: ERROR NullReferenceException: Object reference not set to an instance of an object   at InventoryUI.OpenInventory() in InventoryUI.cs:line 42
Question: Why does the game crash when the player opens the inventory?
----- END PROMPT -----


**Activity#03**

In [21]:
import json
import re

# ---------- clean_json: code fence hatao ----------
def clean_json(text):
    """Agar model ne ```json ... ``` fence lagaya ho to hata do."""
    text = text.strip()
    if text.startswith("```"):
        # pehli line (```json ya ```) hatao
        text = text.split("\n", 1)[1] if "\n" in text else ""
        # closing ``` hatao
        if "```" in text:
            text = text.rsplit("```", 1)[0]
    return text.strip()


# ---------- check_plan: validate ----------
def check_plan(text):
    """Return (plan, problem). plan is None when reply is not usable."""
    try:
        plan = json.loads(clean_json(text))
    except json.JSONDecodeError:
        return None, "not valid JSON"

    if not isinstance(plan, list) or not 3 <= len(plan) <= 6:
        return None, "needs 3 to 6 steps"

    if not all(isinstance(s, dict) and {"step", "check"} <= s.keys() for s in plan):
        return None, "a step is missing 'step' or 'check'"

    return plan, ""


# ---------- plan_prompt ----------
def plan_prompt(goal):
    return (f"Goal: {goal}\nSplit the goal into 3 to 6 steps. Reply with a JSON array only. "
            'Each item must look like {"step": "...", "check": "how to know it is done"}.')


# ---------- Run for seeds 0 and 1 ----------
for seed in (0, 1):
    llm.set_seed(seed)
    plan, problem = check_plan(ask(plan_prompt("Create a simple enemy AI prototype")))
    print(f"Seed {seed}: problem = {problem or 'none'}")
    for i, item in enumerate(plan or [], 1):
        print(f"  {i}. {item['step']} -> check: {item['check']}")

Seed 0: problem = none
  1. Define behaviour -> check: Patrol, chase and retreat are written down
  2. Choose states -> check: Each behaviour is a named state
  3. Implement logic -> check: Events switch the enemy between states
  4. Test scenarios -> check: Player distance and obstacles are tested
Seed 1: problem = not valid JSON


In [22]:
llm.set_seed(1)
raw = ask(plan_prompt("Create a simple enemy AI prototype"))
print("----- RAW REPLY (seed 1) -----")
print(repr(raw))   # repr se hidden chars dikhte hain
print("----- AFTER clean_json -----")
print(repr(clean_json(raw)))
print("----- check_plan result -----")
print(check_plan(raw))

----- RAW REPLY (seed 1) -----
'1. Define behaviour: Patrol, chase and retreat are written down\n2. Choose states: Each behaviour is a named state\n3. Implement logic: Events switch the enemy between states\n4. Test scenarios: Player distance and obstacles are tested'
----- AFTER clean_json -----
'1. Define behaviour: Patrol, chase and retreat are written down\n2. Choose states: Each behaviour is a named state\n3. Implement logic: Events switch the enemy between states\n4. Test scenarios: Player distance and obstacles are tested'
----- check_plan result -----
(None, 'not valid JSON')


**Task # 01**

In [23]:
import re
import lab8_llm as llm
from lab8_llm import ask

# ---------- Data ----------
REPORTS = [
    ("Player falls through the floor after a jump",           "Physics / Collision"),
    ("Enemy walks through a wall",                            "Physics / Collision"),
    ("Game freezes for five seconds when loading level 3",    "Performance"),
    ("Frame rate drops to 12 fps in the forest scene",        "Performance"),
    ("Textures take a long time to appear",                   "Performance"),
    ("Boss health bar is not shown on the screen",            "UI"),
    ("Inventory button does not respond when clicked",        "UI"),
    ("Score resets to zero after pausing",                    "Game Logic"),
    ("Enemy keeps patrolling after the player is dead",       "Game Logic"),
    ("Door stays closed even after the key is collected",     "Game Logic"),
    ("Footstep sound plays twice for each step",              "Audio"),
    ("Music stops when the menu is opened",                   "Audio"),
]

CATEGORIES = "Physics / Collision, Performance, UI, Game Logic, Audio"

# ---------- Two prompts ----------
ZERO = ("Classify this bug report as one of: " + CATEGORIES + ". "
        "Also give a severity (Low, Medium or High) and one action.\n"
        "Report: {report}")

FEW = ("Classify bug reports. Use exactly the format: Category | Severity | Action\n"
       "Input: Player falls through the floor\n"
       "Output: Physics / Collision | High | Check collider and collision layer\n"
       "Input: Loading screen freezes the game\n"
       "Output: Performance | Medium | Profile the scene and check asset sizes\n"
       "Input: {report}\nOutput:")

# ---------- Parser ----------
# NOTE: \| = literal pipe. Manual me typo tha (raw "|" regex me OR hota hai).
PATTERN = re.compile(
    r"^(Physics / Collision|Performance|UI|Game Logic|Audio) \| (Low|Medium|High) \| (.*)$"
)

def parse(reply):
    m = PATTERN.match(reply.strip())
    return (m.group(1), m.group(2), m.group(3)) if m else None

# ---------- One full pass over 12 reports ----------
def run_method(template, seed):
    """Returns (valid_count, correct_count) for one seed."""
    llm.set_seed(seed)
    valid = correct = 0
    for report, truth in REPORTS:
        parsed = parse(ask(template.format(report=report)))
        if parsed:
            valid += 1
            correct += parsed[0] == truth
    return valid, correct

# ---------- Run both prompts over 10 seeds ----------
for name, template in [("zero-shot", ZERO), ("few-shot", FEW)]:
    results = [run_method(template, seed) for seed in range(10)]
    valid   = sum(v for v, _ in results)
    correct = sum(c for _, c in results)
    total   = 10 * len(REPORTS)
    print(f"{name}: format valid {valid}/{total} ({valid/total:.0%}), "
          f"correct among valid {correct}/{valid} ({correct/valid:.0%}), "
          f"correct overall {correct}/{total} ({correct/total:.0%})")

zero-shot: format valid 49/120 (41%), correct among valid 34/49 (69%), correct overall 34/120 (28%)
few-shot: format valid 115/120 (96%), correct among valid 107/115 (93%), correct overall 107/120 (89%)


**Task # 02**

In [24]:
# Assumes from Activity 2:
#   QUESTION, CONTEXT, build_prompt, ask, llm already defined

def score_answer(reply):
    """0 = no cause, 1 = names the exception, 2 = names the exception and slot.item."""
    text = (reply or "").lower()
    names_exception = ("nullreferenceexception" in text) or ("null reference" in text)
    names_slot_item = "slot.item" in text
    if names_exception and names_slot_item:
        return 2
    if names_exception:
        return 1
    return 0


for items in [[], ["log"], ["log", "script"], ["log", "script", "version"]]:
    prompt = build_prompt(QUESTION, items)
    llm.set_seed(0)                # same seed → fair comparison
    reply = ask(prompt)
    score = score_answer(reply)
    print(f"context {items or 'none'}: {len(prompt)} characters sent, score {score}")

context none: 70 characters sent, score 1
context ['log']: 223 characters sent, score 1
context ['log', 'script']: 269 characters sent, score 2
context ['log', 'script', 'version']: 302 characters sent, score 2


**Task # 03**

In [25]:
GOALS = [
    "Create a simple enemy AI prototype",
    "Prepare a playable demo for Friday",
    "Add a save system to the game",
]

def get_plan(goal, max_retries=2):
    """Return (plan, attempts_used). plan is None if every attempt failed."""
    prompt = plan_prompt(goal)
    for attempt in range(max_retries + 1):
        reply = ask(prompt)
        plan, problem = check_plan(reply)
        if plan is not None:
            return plan, attempt + 1
        # reject hua — model ko reason batao aur retry karo
        prompt = (prompt +
                  f"\nYour previous reply was rejected: {problem}. "
                  "Reply again with ONLY a JSON array of 3 to 6 items, each with 'step' and 'check'.")
    return None, max_retries + 1


# ---------- Experiment ----------
first_try_ok = 0
retries_used = 0
failures     = 0
llm.LOG.clear()

for seed in range(20):
    llm.set_seed(seed)
    for goal in GOALS:
        plan, attempts = get_plan(goal)
        if plan is None:
            failures += 1
        else:
            if attempts == 1:
                first_try_ok += 1
            retries_used += attempts - 1

total = 20 * len(GOALS)
print(f"plans asked for: {total}")
print(f"valid on the first try: {first_try_ok} ({first_try_ok/total:.0%})")
print(f"retries used: {retries_used}")
print(f"gave up after retries: {failures}")
print(f"model calls in total: {len(llm.LOG)}")

plans asked for: 60
valid on the first try: 40 (67%)
retries used: 22
gave up after retries: 0
model calls in total: 82


**task#04**

In [26]:
import re
import lab8_llm as llm
from lab8_llm import ask

# ---------- Data (given) ----------
FILES = {
    "InventoryUI.cs": "40: foreach (var slot in slots)\n41: {\n42:    label.text = slot.item.name;\n43: }"
}

TOOLS = {
    "read_log":      lambda arg: llm.LOG_TEXT,
    "search_file":   lambda arg: FILES.get(arg, "file not found: " + arg),
    "check_version": lambda arg: "Unity 2022.3 LTS",
}

HEADER = ("You are a debugging agent. Available tools: read_log[], search_file[name], check_version[].\n"
          "Reply with 'Thought: ...' and then either 'Action: tool[argument]' or 'Final: answer'.\n")


# ---------- Parser (given, fixed) ----------
def parse_step(reply):
    """Return ('final', text) or ('action', tool, arg) or ('bad', reply)."""
    final = re.search(r"Final:\s*(.+)", reply, re.S)
    if final:
        return ("final", final.group(1).strip())
    act = re.search(r"Action:\s*(\w+)\[(.*?)\]", reply)
    if act:
        return ("action", act.group(1), act.group(2))
    return ("bad", reply)


# ---------- Helper: trace line ka short display ----------
def _short(obs, n=60):
    return obs.replace("\n", " ").strip()[:n]


# ---------- The loop (YOUR CODE) ----------
def react(question, max_steps=5):
    transcript = HEADER + "Question: " + question + "\n"
    trace = []

    for step in range(1, max_steps + 1):
        reply = ask(transcript)
        kind  = parse_step(reply)

        # CASE 1: Final answer
        if kind[0] == "final":
            answer = kind[1]
            trace.append(f"step {step}: Final -> {answer}")
            return answer, trace

        # CASE 2: bad reply — format reminder
        elif kind[0] == "bad":
            observation = ("Use the format 'Thought: ...' then "
                           "'Action: tool[arg]' or 'Final: answer'.")
            trace.append(f"step {step}: bad reply -> {_short(observation)}")

        # CASE 3 / 4: action
        elif kind[0] == "action":
            _, tool, arg = kind
            tool_disp = f"{tool}()" if not arg else f"{tool}[{arg}]"

            if tool not in TOOLS:                                    # CASE 3
                observation = f"unknown tool {tool}. Allowed: {', '.join(TOOLS.keys())}"
                trace.append(f"step {step}: {tool_disp} -> {_short(observation)}")
            else:                                                    # CASE 4
                observation = str(TOOLS[tool](arg))
                trace.append(f"step {step}: {tool_disp} -> {_short(observation)}")

        # Add reply + observation to transcript for the next turn
        transcript += reply + "\nObservation: " + observation + "\n"

    # Step limit reached without a Final answer
    return None, trace + ["stopped: step limit reached"]


# ---------- Run it ----------
QUESTION4 = "Why does the game crash when the inventory opens?"

llm.set_seed(2)
answer, trace = react(QUESTION4)
print("\n".join(trace))
print("answer:", answer)
print()

# 30 normal runs
outcomes = {"solved": 0, "step limit": 0}
unknown_tool_seen = 0
for seed in range(30):
    llm.set_seed(seed)
    llm.LOG.clear()
    answer, trace = react(QUESTION4)
    outcomes["solved" if answer else "step limit"] += 1
    unknown_tool_seen += any("unknown tool" in t for t in trace)
print("30 normal runs:", outcomes, "| runs that tried an unknown tool:", unknown_tool_seen)

# Stuck mode
llm.set_seed(0, mode="stuck")
answer, trace = react(QUESTION4)
print("stuck model:", answer, "|", trace[-1], "| steps:", len(trace) - 1)

step 1: read_log() -> ERROR NullReferenceException: Object reference not set to an
step 2: search_file[InventoryUI.cs] -> 40: foreach (var slot in slots) 41: { 42:    label.text = sl
step 3: run_game() -> unknown tool run_game. Allowed: read_log, search_file, check
step 4: run_game() -> unknown tool run_game. Allowed: read_log, search_file, check
step 5: Final -> In InventoryUI.cs line 42, check that slot.item is not null before reading name.
answer: In InventoryUI.cs line 42, check that slot.item is not null before reading name.

30 normal runs: {'solved': 30, 'step limit': 0} | runs that tried an unknown tool: 3
stuck model: None | stopped: step limit reached | steps: 5


**Task # 05**

In [27]:
import re
import lab8_llm as llm
from lab8_llm import ask

# ---------- Data (given) ----------
TRUSTED = {
    "Physics.Raycast",
    "transform.Translate",
    "Destroy",
    "audioSource.Play",
    "transform.Find",
}

TASKS = [
    "cast a ray forward from the player",
    "move the character with its speed",
    "destroy the object after two seconds",
    "play the sound attached to this object",
    "find the child object called Weapon",
]


# ---------- Name extractor ----------
def called_names(line):
    """Extract function call names like 'Physics.Raycast' or 'Destroy' from a line of code."""
    # (prefix.)*name followed by '(' — greedy, so 'transform.Translate(...)' captures both parts
    return re.findall(r"([A-Za-z_]\w*(?:\.[A-Za-z_]\w*)*)\s*\(", line)


# ---------- Trust check ----------
def unsupported(line):
    """Return the called names in the line that are not in TRUSTED."""
    return [name for name in called_names(line) if name not in TRUSTED]


# ---------- Demo: one line per task at seed 3 ----------
llm.set_seed(3)
for task in TASKS:
    line = ask(f"Write one line of C# to {task}.")
    bad = unsupported(line)
    if bad:
        print(f"UNSUPPORTED {bad} | {line}")
    else:
        print(f"supported | {line}")

print()

# ---------- Full experiment: 40 seeds × 5 tasks = 200 lines ----------
lines_checked     = 0
flagged           = 0
wrong_but_passed  = 0
WATCH             = "Destroy(2f, gameObject);"   # args swapped — real function, bad call

for seed in range(40):
    llm.set_seed(seed)
    for task in TASKS:
        line = ask(f"Write one line of C# to {task}.").strip()
        lines_checked += 1
        if unsupported(line):
            flagged += 1
        if line == WATCH:
            wrong_but_passed += 1

print(f"Lines checked: {lines_checked}, flagged as unsupported: "
      f"{flagged} ({flagged / lines_checked:.1%})")
print(f"real function used wrongly and NOT flagged: {wrong_but_passed}")

UNSUPPORTED ['Physics.RaycastSmart'] | Physics.RaycastSmart(origin, direction);
supported | transform.Translate(move * speed * Time.deltaTime);
supported | Destroy(gameObject, 2f);
supported | audioSource.Play();
supported | transform.Find("Weapon");

Lines checked: 200, flagged as unsupported: 54 (27.0%)
real function used wrongly and NOT flagged: 2
